## 6.1 Simple RNN反向传播 - 反向传播流程

#### 1. 这一节我们要解决什么问题 🎯

前面我们已经学习了：

* RNN 的网络结构
* RNN 的前向传播
* RNN 中的矩阵计算和维度变化

现在我们要开始学习一个新的核心知识点：

> RNN 是如何训练的？

而想让 RNN 学会从数据中不断调整参数，本质上就必须回答一个问题：

> 损失函数对各个参数的梯度是怎么来的？

这正是反向传播（Backpropagation）要解决的事情。

但是 RNN 的反向传播和我们以前学习的 MLP 不完全一样，因为在 RNN 中：

* 数据不仅沿着“层”传播
* 还沿着“时间”传播

所以 RNN 的反向传播会比普通神经网络多一条重要路径：

> 梯度不仅要沿网络层回传，还要沿时间维度回传。

这一节我们主要解决 3 个问题：

* RNN 的反向传播整体路径到底是什么？
* 为什么隐藏状态的梯度会来自多个方向？
* 为什么说 RNN 的反向传播本质上是“沿时间展开后再做链式法则”？


#### 2. 先从整体上理解：什么是 RNN 的反向传播？

##### 2.1 普通神经网络中的反向传播

在普通前馈网络中，例如 MLP，前向传播的路径通常是：

输入层 $\rightarrow$ 隐藏层 $\rightarrow$ 输出层 $\rightarrow$ 损失函数

所以反向传播时，梯度就会沿着相反方向回传：

损失函数 $\rightarrow$ 输出层 $\rightarrow$ 隐藏层 $\rightarrow$ 输入层

也就是说，普通网络里的反向传播主要是：

> 沿着网络层的方向，从后往前传递梯度。

##### 2.2 RNN 的反向传播为什么更特殊？

RNN 比普通网络多了一个核心特点：

> 当前时刻的隐藏状态 $h_t$，不仅影响当前时刻的输出，还会继续传给下一时刻。

也就是说，在前向传播时，RNN 有两条主线：

* 第一条：RNN层和全连接层之间的传播  
  在单个时间步内部：  
  $x_t \rightarrow RNN层 \rightarrow h_t \rightarrow 输出层 \rightarrow y_t \rightarrow L_t$
* 第二条：时间之间的传播  
  在不同时间步之间：  
  $h_0 \rightarrow h_1 \rightarrow h_2 \rightarrow h_3 \rightarrow \dots \rightarrow h_T$

所以到了反向传播时，梯度自然也会有两条回传路径：

* 一条沿着 RNN层和全连接层回传
* 一条沿着“时间”回传

##### 2.3 所以什么叫 BPTT？

RNN 的反向传播通常单独叫做：

> BPTT（Backpropagation Through Time）

它的意思就是：

> 把 RNN 沿时间展开，然后在展开后的计算图上做反向传播。

这里的关键词是：

* Backpropagation：反向传播
* Through Time：沿时间展开后传播

所以：

> RNN 的反向传播，本质上还是链式法则；只不过梯度不只沿层回传，还沿时间回传。

#### 3. 先回顾前向传播结构

##### 3.1 单个时间步的前向传播

在第 $t$ 个时间步，最基础的 Simple RNN 前向传播可以写成两部分：

* 第一部分：RNN层计算当前隐藏状态  
  $h_t = f(W_{xh}x_t + W_{hh}h_{t-1} + b_h)$
* 第二部分：输出层根据隐藏状态得到输出  
  $o_t = W_{hy}h_t + b_y$

如果继续写出预测值，则通常还会有：

$\hat{y}_t = g(o_t)$

其中：

* $x_t$：当前时间步输入
* $h_{t-1}$：上一时刻隐藏状态
* $h_t$：当前时刻隐藏状态
* $o_t$：输出层线性结果
* $\hat{y}_t$：当前时刻预测值

##### 3.2 整个序列的总损失

如果序列长度为 $T$，那么总损失通常写成：

$L = \sum_{t=1}^{T} L_t$

也就是说：

> 整个序列的损失 = 每个时间步损失的总和

这一点非常重要，因为它意味着：

* 每个时间步都会对总损失有贡献
* 每个时间步都会对参数产生梯度
* 最后参数更新时，要把所有时间步的影响加起来

#### 4. RNN 反向传播和普通神经网络最大的区别

##### 4.1 普通前馈网络怎么反向传播？

普通网络中，梯度主要沿着“层”往回传：

输出层 $\rightarrow$ 隐藏层 $\rightarrow$ 输入层

这是“空间上的反向传播”。

在 MLP 中，如果你有两层网络：

$x \rightarrow z^{[1]} \rightarrow a^{[1]} \rightarrow z^{[2]} \rightarrow a^{[2]} \rightarrow L$

那么反向传播时，只需要沿着这条链从后往前走：

$L \rightarrow a^{[2]} \rightarrow z^{[2]} \rightarrow a^{[1]} \rightarrow z^{[1]}$

所以 MLP 的反向传播，本质上只是在做：

> 沿层的反向传播。

##### 4.2 RNN 多了一条“时间链”

RNN 不一样。

因为当前隐藏状态 $h_t$ 在前向传播中会被两个地方使用：

* 第一，当前输出层会用它  
  $h_t \rightarrow o_t \rightarrow \hat{y}_t \rightarrow L_t$
* 第二，下一时刻隐藏层还会用它  
  $h_t \rightarrow a_{t+1} \rightarrow h_{t+1}$

所以对于 $h_t$ 来说，它不只是影响当前时刻损失 $L_t$，还会继续影响：

* $L_{t+1}$
* $L_{t+2}$
* $\dots$
* $L_T$

这就导致：

> RNN 中一个中间状态的梯度，不再只来自当前层，而会来自未来时间步。

这就是 RNN 反向传播最本质的特殊之处。


#### 5. 隐藏状态 $h_t$ 为什么是反向传播中的核心

##### 5.1 因为隐藏状态是时间链上的“桥梁”

在整个 RNN 中，真正把不同时间步串起来的，不是输出 $\hat{y}_t$，也不是损失 $L_t$，而是：

$h_0 \rightarrow h_1 \rightarrow h_2 \rightarrow \dots \rightarrow h_T$

所以反向传播时，真正承担“把未来误差传回过去”这个任务的，也是隐藏状态。

也就是说：

> 隐藏状态是 RNN 反向传播中最关键的中间桥梁。

##### 5.2 隐藏状态梯度 $h_t$ 的“汇合”

对于中间某个时间步的隐藏状态 $h_t$，它在前向传播中被两个方向使用：

* 路径一：当前输出层  
  $h_t \rightarrow o_t \rightarrow \hat{y}_t \rightarrow L_t$  
  这条路径说明：  
  当前隐藏状态会影响当前时刻的损失。
* 路径二：下一时刻隐藏层  
  $h_t \rightarrow a_{t+1} \rightarrow h_{t+1} \rightarrow \dots \rightarrow L_{t+1}, \dots, L_T$  
  这条路径说明：  
  当前隐藏状态还会影响未来时刻的损失。

所以：

> 总损失对 $h_t$ 的梯度，不能只看当前时刻，还必须把未来时间步倒流回来的梯度也算进去。

##### 5.3 所以 $h_t$ 的总梯度是什么？

从流程上理解，可以写成：

> $h_t$ 的总梯度 = 当前时刻输出路径传回来的梯度 + 未来时间步沿时间链传回来的梯度

这其实就是计算图里的经典规则：

> 一个节点如果流向多个后续节点，那么反向传播时，各条分支传回来的梯度要相加。

所以 RNN 中隐藏状态梯度的“汇合”，本质上并不是新规则，而是：

> 链式法则在时间展开图中的自然结果。

#### 6. RNN 反向传播的总体路径应该怎样理解？

现在我们再从参数角度看，会更清楚为什么隐藏状态这么重要。

在 RNN 中，我们最终真正想求的是这些参数的梯度：

* $\frac{\partial L}{\partial W_{xh}}$
* $\frac{\partial L}{\partial W_{hh}}$
* $\frac{\partial L}{\partial b_h}$
* $\frac{\partial L}{\partial W_{hy}}$
* $\frac{\partial L}{\partial b_y}$

##### 6.1 第一部分：输出层往回传

每个时间步都有一条本地的输出路径：

$L_t \rightarrow o_t \rightarrow h_t$

这一部分和普通 MLP 非常像。

它主要负责：

* 从损失函数算出输出层误差
* 得到输出层参数的梯度
* 把当前时刻输出层的误差信号传回 $h_t$

对应参数是：

* $W_{hy}$
* $b_y$

它们只和当前时刻的隐藏状态 $h_t$ 有关，所以这部分反向传播和 MLP 很像。

##### 6.2 第二部分：隐藏状态沿时间往回传

因为 $h_t$ 还会影响下一时刻，所以当前时刻的隐藏状态梯度还会继续向前一时刻传播：

$h_t \rightarrow h_{t-1} \rightarrow h_{t-2} \rightarrow \dots$

更准确地说，是通过：

$h_t \rightarrow a_{t+1} \rightarrow h_{t+1}$

在反向时把未来误差倒流回来。

所以你可以把它理解成：

> 未来时间步的梯度，会借助隐藏状态链，一步一步倒流回过去。

##### 6.3 所以完整流程是

对于某个时间步 $t$，反向传播要同时考虑：

* 当前输出层传回来的梯度
* 下一时刻沿时间链传回来的梯度

然后再继续往前传播。

所以 RNN 反向传播的整体感觉应该是：

> 每个时间步都像在“接收两股梯度”，然后再把合并后的梯度继续往前传。

具体可以理解为：

* 先从输出层回到 $h_t$
* 同时未来时间步的梯度也会回到 $h_t$
* 这两部分梯度在 $h_t$ 处汇合
* 再继续往前传到 $h_{t-1}$

##### 6.4 整体数据流
```
前向：  x_t → [a_t] → h_t → [o_t] → L_t
                ↑               
              h_{t-1}           

反向（每个时间步）：

L_t → δ_t^o → ∂L/∂W_hy（直接更新）
           ↓
        ∂L/∂h_t 的"直接部分" = W_hy^T δ_t^o
           +
        ∂L/∂h_t 的"来自未来部分" = W_hh^T δ_{t+1}^h
           ↓  ⊙ φ'(a_t)
        δ_t^h  → ∂L/∂W_hh, ∂L/∂W_xh（累加）
           ↓  × W_hh^T
        传给 δ_{t-1}^h（继续往前）
```